# Annotated ID Video Renderer

Draws player bounding boxes, team colours, track IDs, and frame numbers onto the match video.
Encodes output with **NVENC** (T4 GPU) for fast render.

**Before running:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
VIDEO_PATH   = "/content/HIL-HAZ_half2.mp4"                       # input video
TRACKS_CSV   = "/content/per_frame_tracks_HIL-HAZ_half2.csv"      # tracker output CSV
ANNO_OUT     = "/content/annotated_ids.mp4"                        # output video

FRAME_START  = 0      # first frame to render  (0 = from beginning)
FRAME_END    = None   # last frame  (None = full video)
RENDER_EVERY = 1      # 1 = every frame · 2 = every other frame (2× faster, half fps)

VIDEO_BITRATE = "8M"  # NVENC output bitrate  (increase for higher quality)

In [ ]:
# ── GPU check + minimal deps ──────────────────────────────────────────────────
import subprocess, sys

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True)
if gpu.returncode == 0:
    print("GPU:", gpu.stdout.strip())
else:
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to Runtime → Change runtime type → T4 GPU, then re-run.")

# Check NVENC is available via ffmpeg
nvenc = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"],
                       capture_output=True, text=True)
if "h264_nvenc" not in nvenc.stdout:
    print("⚠ h264_nvenc not in this ffmpeg build — installing ffmpeg with NVENC support.")
    subprocess.run(["apt-get", "install", "-qq", "ffmpeg"], check=True)
else:
    print("NVENC: h264_nvenc available ✓")

# Ensure pandas + tqdm are present
for pkg in ["pandas", "tqdm"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Setup OK.")

In [ ]:
# ── Render annotated video ────────────────────────────────────────────────────
import cv2, os, subprocess
import numpy as np
import pandas as pd
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x

# ── class / colour constants ──────────────────────────────────────────────────
BALL_CLASS   = 0
GK_CLASS     = 1
PLAYER_CLASS = 2
REF_CLASS    = 3
PEOPLE       = (GK_CLASS, PLAYER_CLASS, REF_CLASS)

_COL = {
    (PLAYER_CLASS,  0): (240, 100, 180),   # home  → pink
    (PLAYER_CLASS,  1): ( 30, 210, 240),   # away  → cyan
    (PLAYER_CLASS, -1): (160, 160, 160),   # unknown team
    (GK_CLASS,      0): (180,  60, 255),   # home GK → purple
    (GK_CLASS,      1): ( 80, 200, 120),   # away GK → green
    (GK_CLASS,     -1): (200, 200,  60),
    (REF_CLASS,    -1): (255, 140,   0),   # referee → orange
}
_CLS_LBL = {PLAYER_CLASS: "P", GK_CLASS: "GK", REF_CLASS: "REF"}
BALL_COL  = (255, 220, 0)

def _col(cls, team):
    return _COL.get((int(cls), int(team)),
           _COL.get((int(cls), -1), (160, 160, 160)))

def _label(frame, text, ox, oy, scale, thick, col):
    (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thick)
    cv2.rectangle(frame, (ox-2, oy-th-bl-2), (ox+tw+2, oy+2), (0, 0, 0), -1)
    cv2.putText(frame, text, (ox, oy), cv2.FONT_HERSHEY_SIMPLEX,
                scale, col, thick, cv2.LINE_AA)

# ── load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv(TRACKS_CSV, encoding="utf-8", encoding_errors="replace")
track_grp = df.set_index("frame")
print(f"CSV: {len(df):,} rows · {df['frame'].nunique()} frames · "
      f"{df['display_track_id'].nunique()} track IDs")

# ── open video ────────────────────────────────────────────────────────────────
cap   = cv2.VideoCapture(VIDEO_PATH)
fps   = cap.get(cv2.CAP_PROP_FPS) or 25.0
tot   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Video: {tot} frames @ {fps:.2f} fps  ({W}×{H})")

f_start  = int(FRAME_START)
f_end    = (int(FRAME_END) if FRAME_END is not None else tot - 1)
f_end    = min(f_end, tot - 1)
out_fps  = fps / max(1, RENDER_EVERY)
sc       = W / 1920   # scale stroke/font to frame resolution

# ── open NVENC pipe via ffmpeg ────────────────────────────────────────────────
os.makedirs(os.path.dirname(ANNO_OUT) or ".", exist_ok=True)
ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-f", "rawvideo", "-vcodec", "rawvideo",
    "-s", f"{W}x{H}",
    "-pix_fmt", "bgr24",
    "-r", str(out_fps),
    "-i", "pipe:0",
    "-c:v", "h264_nvenc",
    "-preset", "fast",
    "-b:v", VIDEO_BITRATE,
    "-pix_fmt", "yuv420p",
    ANNO_OUT
]
proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)

# ── render loop ───────────────────────────────────────────────────────────────
cap.set(cv2.CAP_PROP_POS_FRAMES, f_start)
n_written = 0

for fi in tqdm(range(f_start, f_end + 1), desc="render", unit="f"):
    ok, bgr = cap.read()
    if not ok:
        break
    if (fi - f_start) % RENDER_EVERY != 0:
        continue

    frame = bgr  # draw directly on the decoded frame (no extra copy needed)

    # frame number — top-left
    ts = f"{fi:05d}   {fi/fps:.2f}s"
    cv2.rectangle(frame, (0, 0), (360, 42), (0, 0, 0), -1)
    cv2.putText(frame, ts, (8, 30), cv2.FONT_HERSHEY_SIMPLEX,
                1.0, (255, 220, 0), 2, cv2.LINE_AA)

    if fi in track_grp.index:
        rows = track_grp.loc[[fi]]

        # ball
        for _, br in rows[rows["class_id"] == BALL_CLASS].iterrows():
            cx = int((br.x1 + br.x2) / 2)
            cy = int((br.y1 + br.y2) / 2)
            cv2.circle(frame, (cx, cy), max(6, int(8*sc)), BALL_COL, 2, cv2.LINE_AA)

        # players / GKs / referees
        for _, r in rows[rows["class_id"].isin(PEOPLE)].iterrows():
            tid  = int(r.display_track_id)
            cls  = int(r.class_id)
            team = int(r.team_id) if pd.notna(r.team_id) else -1
            col  = _col(cls, team)
            x1, y1, x2, y2 = int(r.x1), int(r.y1), int(r.x2), int(r.y2)
            bw   = max(1, int(2*sc + 1))

            cv2.rectangle(frame, (x1, y1), (x2, y2), col, bw)

            role = _CLS_LBL.get(cls, "?")
            tm   = "" if team < 0 else f"t{team}"
            _label(frame, str(tid),       x1, max(0, y1-20),
                   0.85*max(sc, 0.6), 2, col)
            _label(frame, f"{role}{tm}",  x1, max(0, y1-2),
                   0.5*max(sc, 0.5),  1, col)

    proc.stdin.write(frame.tobytes())
    n_written += 1

proc.stdin.close()
proc.wait()
cap.release()

size_mb = os.path.getsize(ANNO_OUT) / 1e6
print(f"\nDone — {n_written} frames → {ANNO_OUT}  ({size_mb:.1f} MB)")
print(f"Covers frames {f_start}–{f_end}  ({(f_end-f_start)/fps:.1f}s @ {out_fps:.1f}fps)")

In [ ]:
# ── Download the video ────────────────────────────────────────────────────────
from google.colab import files as colab_files
colab_files.download(ANNO_OUT)

## Extract a frame range → compact clip (for sharing / review)

Pulls a short window of frames out of a video, **downscales + re-compresses** them into a
small MP4 you can upload back to the chat for visual review of a specific ID-swap event.

- Point `SRC_VIDEO` at the annotated render (`ANNO_OUT`) or the raw match video.
- Set `CLIP_START` / `CLIP_END` to the frames you want to inspect (e.g. a swap chain).
- `SCALE_W` and `CRF` control size — the defaults keep a ~hundred-frame clip to a few MB.</cell id="cell-download">

In [ ]:
# ── Extract a frame range and compress it into a small, shareable clip ─────────
import cv2, os, subprocess

# ── Configuration ─────────────────────────────────────────────────────────────
SRC_VIDEO  = ANNO_OUT            # source video (annotated render, or raw e.g. VIDEO_PATH)
CLIP_START = 4270                # first frame to extract
CLIP_END   = 5720                # last frame to extract (inclusive)
CLIP_OUT   = "/content/clip_4270-5720.mp4"

SCALE_W    = 1280                # downscale width in px (-1 keeps source width)
CRF        = 20                  # x264 quality: lower = bigger/sharper file (18-23 = high quality, ~10-15MB here)
CLIP_FPS   = None                # None = use source fps; or set e.g. 12 for smaller file

# ── pull the frame range out of the source video ──────────────────────────────
cap = cv2.VideoCapture(SRC_VIDEO)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open {SRC_VIDEO}")
src_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
tot     = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_fps = CLIP_FPS or src_fps
c_end   = min(int(CLIP_END), tot - 1)
print(f"Source: {tot} frames @ {src_fps:.2f} fps ({W}×{H}) → extracting {CLIP_START}–{c_end}")

# scaled output dimensions (keep even numbers for yuv420p)
if SCALE_W and SCALE_W > 0:
    ow = int(SCALE_W); ow -= ow % 2
    oh = int(round(H * ow / W)); oh -= oh % 2
else:
    ow, oh = W - W % 2, H - H % 2

# ── pipe the selected frames straight into ffmpeg (libx264 CRF = small file) ───
os.makedirs(os.path.dirname(CLIP_OUT) or ".", exist_ok=True)
ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-f", "rawvideo", "-vcodec", "rawvideo",
    "-s", f"{W}x{H}", "-pix_fmt", "bgr24", "-r", str(out_fps),
    "-i", "pipe:0",
    "-vf", f"scale={ow}:{oh}",
    "-c:v", "libx264", "-preset", "veryslow",
    "-crf", str(CRF), "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    CLIP_OUT,
]
proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)

cap.set(cv2.CAP_PROP_POS_FRAMES, int(CLIP_START))
n = 0
for fi in range(int(CLIP_START), c_end + 1):
    ok, bgr = cap.read()
    if not ok:
        break
    proc.stdin.write(bgr.tobytes())
    n += 1
proc.stdin.close()
proc.wait()
cap.release()

size_mb = os.path.getsize(CLIP_OUT) / 1e6
print(f"\nDone — {n} frames → {CLIP_OUT}  ({ow}×{oh} @ {out_fps:.0f}fps · {size_mb:.2f} MB)")
if size_mb > 25:
    print("⚠ File is large — raise CRF (e.g. 24-26), lower SCALE_W (e.g. 960), "
          "or set CLIP_FPS=15 to shrink it further before uploading.")
elif size_mb < 8:
    print("⚠ File is smaller than expected — lower CRF (e.g. 16-18) or raise SCALE_W "
          "if you need more detail/size.")

# download it so you can drag it into the chat
try:
    from google.colab import files as colab_files
    colab_files.download(CLIP_OUT)
except Exception:
    print(f"(Not on Colab — grab the file at {CLIP_OUT})")